# CascadeFlow VLM Routing Experiment

**Goal**: Evaluate cascade flow model selection on our test dataset and measure cost vs accuracy tradeoffs.

## Experiment Overview
1. Load test dataset from final_dataset/router_pivot_dataset_test.parquet
2. Configure cascade with our 5 VLMs running on vLLM servers
3. Run samples through cascade flow
4. Evaluate accuracy and cost metrics
5. Visualize cascade behavior and cost savings

## Available Models (all self-hosted on vLLM)
- Port 8805: PatronusAI/glider (evaluator/judge model)
- Port 8804: deepseek-ai/DeepSeek-OCR (OCR specialist)
- Port 8803: Qwen/Qwen2.5-VL-3B-Instruct (small, fast)
- Port 8802: Qwen/Qwen2.5-VL-7B-Instruct (medium)
- Port 8801: Qwen/Qwen3-VL-8B-Thinking (reasoning model)
- Port 8800: google/gemma-3-27b-it (large, expensive)

## 1. Imports and Setup

In [1]:
! uv pip install seaborn

Using Python 3.14.0 environment at: cascadeflow_env
Audited 1 package in 16ms


In [2]:
import asyncio
import json
import time
from pathlib import Path
from typing import Dict, List, Optional, Any
from dataclasses import dataclass, asdict
from datetime import datetime

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
import httpx

# CascadeFlow imports
from cascadeflow import CascadeAgent, ModelConfig
from cascadeflow.telemetry import CostTracker, MetricsCollector

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print('✓ Imports successful')

✓ Imports successful


## 2. Configuration

In [3]:
# Paths
PROJECT_ROOT = Path.cwd().parent.parent
DATASET_DIR = PROJECT_ROOT / 'dataset' / 'final_dataset' / 'router_final'
TEST_DATASET_PATH = DATASET_DIR / 'router_test_final.parquet'
OUTPUT_DIR = Path.cwd() / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

print(f'Project root: {PROJECT_ROOT}')
print(f'Dataset path: {TEST_DATASET_PATH}')
print(f'Dataset exists: {TEST_DATASET_PATH.exists()}')
print(f'Output dir: {OUTPUT_DIR}')

# Experiment settings
MAX_SAMPLES = None  # Set to None to run on entire test set
RANDOM_SEED = 42

# Model cost estimates (cost per 1K tokens)
# Since all models are self-hosted, we estimate relative compute cost
# based on model size and latency for cost comparison
MODEL_COSTS = {
    'Qwen/Qwen2.5-VL-3B-Instruct': 0.00001,   # Smallest, fastest
    'deepseek-ai/DeepSeek-OCR': 0.00002,       # Small OCR specialist
    'Qwen/Qwen2.5-VL-7B-Instruct': 0.00005,    # Medium
    'Qwen/Qwen3-VL-8B-Thinking': 0.00008,      # Reasoning model
    'google/gemma-3-27b-it': 0.0002,           # Largest, most expensive
}

# Model server endpoints
MODEL_ENDPOINTS = {
    'Qwen/Qwen2.5-VL-3B-Instruct': 'http://localhost:8803/v1',
    'deepseek-ai/DeepSeek-OCR': 'http://localhost:8804/v1',
    'Qwen/Qwen2.5-VL-7B-Instruct': 'http://localhost:8802/v1',
    'Qwen/Qwen3-VL-8B-Thinking': 'http://localhost:8801/v1',
    'google/gemma-3-27b-it': 'http://localhost:8800/v1',
}

print(f'\nExperiment settings:')
print(f'  Max samples: {MAX_SAMPLES if MAX_SAMPLES else "All"}')
print(f'  Random seed: {RANDOM_SEED}')

Project root: /Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/Which_VLM_Router
Dataset path: /Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/Which_VLM_Router/dataset/final_dataset/router_final/router_test_final.parquet
Dataset exists: True
Output dir: /Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/Which_VLM_Router/code_base/cascadeflow/outputs

Experiment settings:
  Max samples: All
  Random seed: 42


## 3. Health Check for Model Servers

In [4]:
async def check_model_health(base_url: str, model_name: str) -> tuple[bool, Optional[str]]:
    """Check if a model server is healthy and return the actual model ID."""
    try:
        async with httpx.AsyncClient(timeout=10.0) as client:
            response = await client.get(f"{base_url}/models")
            
            if response.status_code != 200:
                return False, None
            
            data = response.json()
            models = [m.get('id', '') for m in data.get('data', [])]
            
            if models:
                return True, models[0]
            return False, None
    except Exception as e:
        print(f"Error checking {base_url}: {e}")
        return False, None

# Run health checks
print('Checking model server health...\n')
health_status = {}

for model_name, base_url in MODEL_ENDPOINTS.items():
    is_healthy, actual_model_id = await check_model_health(base_url, model_name)
    health_status[model_name] = {
        'healthy': is_healthy,
        'model_id': actual_model_id,
        'base_url': base_url
    }
    
    status_icon = '✅' if is_healthy else '❌'
    print(f'{status_icon} {model_name}')
    print(f'   URL: {base_url}')
    if is_healthy:
        print(f'   Model ID: {actual_model_id}')
    print()

# Check if we have enough models to run
healthy_models = [m for m, status in health_status.items() if status['healthy']]
print(f'\nHealthy models: {len(healthy_models)}/{len(MODEL_ENDPOINTS)}')

if len(healthy_models) < 2:
    print('\n⚠️  Warning: Need at least 2 healthy models to run cascade')
else:
    print('\n✓ Ready to proceed with cascade setup')

Checking model server health...

✅ Qwen/Qwen2.5-VL-3B-Instruct
   URL: http://localhost:8803/v1
   Model ID: Qwen/Qwen2.5-VL-3B-Instruct

✅ deepseek-ai/DeepSeek-OCR
   URL: http://localhost:8804/v1
   Model ID: deepseek-ai/DeepSeek-OCR

✅ Qwen/Qwen2.5-VL-7B-Instruct
   URL: http://localhost:8802/v1
   Model ID: Qwen/Qwen2.5-VL-7B-Instruct

✅ Qwen/Qwen3-VL-8B-Thinking
   URL: http://localhost:8801/v1
   Model ID: Qwen/Qwen3-VL-8B-Thinking

✅ google/gemma-3-27b-it
   URL: http://localhost:8800/v1
   Model ID: google/gemma-3-27b-it


Healthy models: 5/5

✓ Ready to proceed with cascade setup


## 4. Load Test Dataset

In [5]:
# Load test dataset
print(f'Loading test dataset from {TEST_DATASET_PATH}...')
test_df = pd.read_parquet(TEST_DATASET_PATH)

print(f'Total test samples: {len(test_df):,}')
print(f'Columns: {len(test_df.columns)}')

# Sample if needed
if MAX_SAMPLES and len(test_df) > MAX_SAMPLES:
    test_df = test_df.sample(n=MAX_SAMPLES, random_state=RANDOM_SEED).reset_index(drop=True)
    print(f'Sampled to: {len(test_df):,} samples')

# Display sample information
print(f'\nDataset info:')
print(f'  Unique tasks: {test_df["router_task"].nunique()}')
print(f'  Unique source configs: {test_df["source_config"].nunique()}')

# Show task distribution
print(f'\nTask distribution:')
task_counts = test_df['router_task'].value_counts()
for task, count in task_counts.head(10).items():
    print(f'  {task}: {count}')

test_df.head(3)

Loading test dataset from /Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/Which_VLM_Router/dataset/final_dataset/router_final/router_test_final.parquet...
Total test samples: 13,707
Columns: 50

Dataset info:
  Unique tasks: 30
  Unique source configs: 48

Task distribution:
  table_reasoning: 1822
  chart_reasoning: 1277
  document_ocr: 960
  general_vqa: 915
  spatial_reasoning: 873
  chart_captioning: 602
  table_math: 588
  scene_text_ocr: 580
  geometry_reasoning: 504
  code_generation: 317


,sample_id,image_path,image_bytes_hash,prompt_raw,img_width,img_height,img_aspect_ratio,txt_prompt_length_chars,txt_prompt_length_words,router_task,...,qwen3_vl_8b_thinking__sample_score,qwen3_vl_8b_thinking__cost,qwen3_vl_8b_thinking__valid_mask,qwen3_vl_8b_thinking__is_correct,qwen3_vl_8b_thinking__score_f1,gemma_3_27b__sample_score,gemma_3_27b__cost,gemma_3_27b__valid_mask,gemma_3_27b__is_correct,gemma_3_27b__score_f1
0,ai2d_00004_bf3d9c5fd30bf304,None,bf3d9c5fd30bf304,Question: If the Termites in the community bel...,1100.0,532.0,2.068,211.0,34.0,diagram_reasoning,...,0.850000,0.001191,True,True,0.000000,0.853030,0.000060,True,True,0.030303
1,ai2d_00007_e6f58451a22503d1,None,e6f58451a22503d1,Question: what does the 2nd picture show?\nCho...,1500.0,1344.0,1.116,162.0,27.0,diagram_reasoning,...,0.853226,0.000758,True,True,0.032258,0.154651,0.000042,True,False,0.046512
2,ai2d_00019_e5f934cfc0951972,None,e5f934cfc0951972,Question: A food web for a ecosystem is shown ...,305.0,297.0,1.027,223.0,40.0,diagram_reasoning,...,-0.050000,0.001104,True,False,0.000000,0.157273,0.000059,True,False,0.072727


## 5. Configure Cascade Flow

We'll create a 3-tier cascade:
- **Tier 1 (Fast)**: Qwen2.5-VL-3B - handles simple queries
- **Tier 2 (Medium)**: Qwen2.5-VL-7B - handles moderate complexity
- **Tier 3 (Premium)**: Gemma-3-27B - handles complex queries requiring best accuracy

In [6]:
# Filter to only healthy models
available_models = [
    model_name for model_name in [
        'Qwen/Qwen2.5-VL-3B-Instruct',
        'Qwen/Qwen2.5-VL-7B-Instruct', 
        'google/gemma-3-27b-it',
        "Qwen/Qwen3-VL-8B-Thinking",
        "deepseek-ai/DeepSeek-OCR",
    ] if model_name in healthy_models
]

print(f'Building cascade with {len(available_models)} models:')
for i, model in enumerate(available_models, 1):
    print(f'  Tier {i}: {model}')

# Create cascade configuration
cascade_models = []
for model_name in available_models:
    config = ModelConfig(
        name=health_status[model_name]['model_id'],
        provider='vllm',
        base_url=health_status[model_name]['base_url'],
        cost=MODEL_COSTS[model_name],
        quality_threshold=0.7,  # Accept if confidence >= 70%
    )
    cascade_models.append(config)

# Create cascade agent
agent = CascadeAgent(models=cascade_models)

print(f'\n✓ Cascade agent created with {len(cascade_models)} tiers')
print(f'\nCascade configuration:')
for i, model in enumerate(cascade_models, 1):
    print(f'  Tier {i}:')
    print(f'    Model: {model.name}')
    print(f'    Cost: ${model.cost:.6f} per 1K tokens')
    print(f'    URL: {model.base_url}')
    print(f'    Quality threshold: {model.quality_threshold}')

LiteLLM not installed. Cost tracking will use fallback estimates. Install with: pip install litellm
LiteLLM not available. Cost calculations will use fallback estimates.
LiteLLM not available. Cost calculations will use fallback estimates.
LiteLLM not available. Cost calculations will use fallback estimates.
LiteLLM not available. Cost calculations will use fallback estimates.
LiteLLM not available. Cost calculations will use fallback estimates.
FastEmbed not available. Install with: pip install fastembed


Building cascade with 5 models:
  Tier 1: Qwen/Qwen2.5-VL-3B-Instruct
  Tier 2: Qwen/Qwen2.5-VL-7B-Instruct
  Tier 3: google/gemma-3-27b-it
  Tier 4: Qwen/Qwen3-VL-8B-Thinking
  Tier 5: deepseek-ai/DeepSeek-OCR

✓ Cascade agent created with 5 tiers

Cascade configuration:
  Tier 1:
    Model: Qwen/Qwen2.5-VL-3B-Instruct
    Cost: $0.000010 per 1K tokens
    URL: http://localhost:8803/v1
    Quality threshold: 0.7
  Tier 2:
    Model: Qwen/Qwen2.5-VL-7B-Instruct
    Cost: $0.000050 per 1K tokens
    URL: http://localhost:8802/v1
    Quality threshold: 0.7
  Tier 3:
    Model: google/gemma-3-27b-it
    Cost: $0.000200 per 1K tokens
    URL: http://localhost:8800/v1
    Quality threshold: 0.7
  Tier 4:
    Model: Qwen/Qwen3-VL-8B-Thinking
    Cost: $0.000080 per 1K tokens
    URL: http://localhost:8801/v1
    Quality threshold: 0.7
  Tier 5:
    Model: deepseek-ai/DeepSeek-OCR
    Cost: $0.000020 per 1K tokens
    URL: http://localhost:8804/v1
    Quality threshold: 0.7


## 6. Setup Evaluation Tracking

In [7]:
@dataclass
class EvaluationResult:
    """Track results for each sample evaluation."""
    sample_id: str
    source_config: str
    router_task: str
    prompt: str
    ground_truth: str
    
    # Cascade results
    model_used: str
    response: str
    cascaded: bool
    draft_accepted: bool
    
    # Performance metrics
    latency_ms: float
    total_cost: float
    total_tokens: int
    
    # Accuracy metrics (comparing to ground truth from dataset)
    is_correct: Optional[bool] = None
    exact_match: Optional[bool] = None
    
    # Additional metadata
    timestamp: str = ''
    error: Optional[str] = None

# Initialize tracking
cost_tracker = CostTracker(
    budget_limit=10.0,  # $10 budget for experiment
    warn_threshold=0.8,
    verbose=False
)

metrics_collector = MetricsCollector()
evaluation_results: List[EvaluationResult] = []

print('✓ Evaluation tracking setup complete')

✓ Evaluation tracking setup complete


## 7. Run Cascade Experiment

In [8]:
async def evaluate_sample(row: pd.Series, agent: CascadeAgent) -> EvaluationResult:
    """Evaluate a single sample through the cascade."""
    try:
        # Run through cascade
        result = await agent.run(
            query=row['prompt_raw'],
            max_tokens=512,
            temperature=0.0,  # Deterministic for evaluation
        )
        
        # Extract metrics
        total_cost = getattr(result, 'total_cost', 0.0)
        total_tokens = getattr(result, 'total_tokens', 0)
        latency_ms = getattr(result, 'latency_ms', 0.0)
        cascaded = getattr(result, 'cascaded', False)
        draft_accepted = getattr(result, 'draft_accepted', False)
        model_used = getattr(result, 'model_used', 'unknown')
        response = getattr(result, 'content', '')
        
        # Simple accuracy check (exact match with ground truth)
        ground_truth = str(row.get('ground_truth', '')).strip().lower()
        response_clean = response.strip().lower()
        exact_match = ground_truth == response_clean
        
        # More lenient check - contains ground truth
        is_correct = ground_truth in response_clean if ground_truth else None
        
        return EvaluationResult(
            sample_id=row['sample_id'],
            source_config=row['source_config'],
            router_task=row.get('router_task', 'unknown'),
            prompt=row['prompt_raw'][:200],  # Truncate for storage
            ground_truth=ground_truth,
            model_used=model_used,
            response=response[:500],  # Truncate for storage
            cascaded=cascaded,
            draft_accepted=draft_accepted,
            latency_ms=latency_ms,
            total_cost=total_cost,
            total_tokens=total_tokens,
            is_correct=is_correct,
            exact_match=exact_match,
            timestamp=datetime.now().isoformat(),
        )
        
    except Exception as e:
        return EvaluationResult(
            sample_id=row['sample_id'],
            source_config=row['source_config'],
            router_task=row.get('router_task', 'unknown'),
            prompt=row['prompt_raw'][:200],
            ground_truth=str(row.get('ground_truth', '')),
            model_used='error',
            response='',
            cascaded=False,
            draft_accepted=False,
            latency_ms=0.0,
            total_cost=0.0,
            total_tokens=0,
            error=str(e),
            timestamp=datetime.now().isoformat(),
        )

# Run evaluation
print(f'Starting cascade evaluation on {len(test_df)} samples...\n')
start_time = time.time()

for idx, row in tqdm(test_df.iterrows(), total=len(test_df), desc='Evaluating'):
    result = await evaluate_sample(row, agent)
    evaluation_results.append(result)
    
    # Track costs
    if result.total_cost > 0:
        cost_tracker.add_cost(
            model=result.model_used,
            provider='vllm',
            tokens=result.total_tokens,
            cost=result.total_cost,
            query_id=result.sample_id,
        )

elapsed_time = time.time() - start_time

print(f'\n✓ Evaluation complete!')
print(f'  Total time: {elapsed_time:.2f}s')
print(f'  Avg time per sample: {elapsed_time/len(test_df):.2f}s')
print(f'  Samples processed: {len(evaluation_results)}')

# Quick summary
successful = [r for r in evaluation_results if r.error is None]
failed = [r for r in evaluation_results if r.error is not None]
print(f'  Successful: {len(successful)}')
print(f'  Failed: {len(failed)}')

Starting cascade evaluation on 13707 samples...



Evaluating:   0%|          | 0/13707 [00:00<?, ?it/s]

CancelledError: 

## 8. Save Results

In [ ]:
# Convert results to DataFrame
results_df = pd.DataFrame([asdict(r) for r in evaluation_results])

# Save to files
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
results_csv = OUTPUT_DIR / f'cascade_results_{timestamp}.csv'
results_json = OUTPUT_DIR / f'cascade_results_{timestamp}.json'

results_df.to_csv(results_csv, index=False)
results_df.to_json(results_json, orient='records', indent=2)

print(f'Results saved:')
print(f'  CSV: {results_csv}')
print(f'  JSON: {results_json}')
print(f'\nResults shape: {results_df.shape}')

results_df.head()

## 9. Analysis & Metrics

In [ ]:
# Filter successful results
success_df = results_df[results_df['error'].isna()].copy()

print('=' * 80)
print('CASCADE FLOW EXPERIMENT RESULTS')
print('=' * 80)

# Overall metrics
print(f'\nOVERALL METRICS:')
print(f'  Total samples: {len(results_df)}')
print(f'  Successful: {len(success_df)} ({len(success_df)/len(results_df)*100:.1f}%)')
print(f'  Failed: {len(results_df) - len(success_df)}')

# Cascade behavior
cascaded_count = success_df['cascaded'].sum()
draft_accepted_count = success_df['draft_accepted'].sum()

print(f'\nCASCADE BEHAVIOR:')
print(f'  Cascaded queries: {cascaded_count} ({cascaded_count/len(success_df)*100:.1f}%)')
print(f'  Draft accepted: {draft_accepted_count} ({draft_accepted_count/len(success_df)*100:.1f}%)')
print(f'  Direct routing: {len(success_df) - cascaded_count} ({(len(success_df)-cascaded_count)/len(success_df)*100:.1f}%)')

# Model usage distribution
print(f'\nMODEL USAGE:')
model_counts = success_df['model_used'].value_counts()
for model, count in model_counts.items():
    pct = count / len(success_df) * 100
    print(f'  {model}: {count} ({pct:.1f}%)')

# Performance metrics
total_cost = success_df['total_cost'].sum()
avg_cost = success_df['total_cost'].mean()
avg_latency = success_df['latency_ms'].mean()
total_tokens = success_df['total_tokens'].sum()

print(f'\nPERFORMANCE METRICS:')
print(f'  Total cost: ${total_cost:.6f}')
print(f'  Avg cost per sample: ${avg_cost:.6f}')
print(f'  Avg latency: {avg_latency:.0f}ms')
print(f'  Total tokens: {total_tokens:,}')

# Accuracy metrics (if available)
if 'is_correct' in success_df.columns:
    correct_count = success_df['is_correct'].sum()
    exact_match_count = success_df['exact_match'].sum()
    
    print(f'\nACCURACY METRICS:')
    print(f'  Contains ground truth: {correct_count}/{len(success_df)} ({correct_count/len(success_df)*100:.1f}%)')
    print(f'  Exact match: {exact_match_count}/{len(success_df)} ({exact_match_count/len(success_df)*100:.1f}%)')

# Task-wise breakdown
print(f'\nTOP TASKS BY VOLUME:')
task_counts = success_df['router_task'].value_counts().head(10)
for task, count in task_counts.items():
    avg_task_cost = success_df[success_df['router_task'] == task]['total_cost'].mean()
    print(f'  {task}: {count} samples, avg cost ${avg_task_cost:.6f}')

print('\n' + '=' * 80)

## 10. Cost Comparison: Cascade vs Always-Best Model

In [ ]:
# Calculate what cost would be if we always used the most expensive model
most_expensive_model = max(available_models, key=lambda m: MODEL_COSTS[m])
most_expensive_cost = MODEL_COSTS[most_expensive_model]

# Estimate baseline cost (always using best/most expensive model)
baseline_cost = success_df['total_tokens'].sum() / 1000 * most_expensive_cost
cascade_cost = success_df['total_cost'].sum()

savings = baseline_cost - cascade_cost
savings_pct = (savings / baseline_cost * 100) if baseline_cost > 0 else 0

print('=' * 80)
print('COST SAVINGS ANALYSIS')
print('=' * 80)
print(f'\nBaseline (always {most_expensive_model}):')
print(f'  Estimated cost: ${baseline_cost:.6f}')
print(f'  Cost per sample: ${baseline_cost/len(success_df):.6f}')

print(f'\nCascade flow:')
print(f'  Actual cost: ${cascade_cost:.6f}')
print(f'  Cost per sample: ${cascade_cost/len(success_df):.6f}')

print(f'\n💰 SAVINGS:')
print(f'  Absolute: ${savings:.6f}')
print(f'  Percentage: {savings_pct:.1f}%')

# Extrapolate to 10K queries
if len(success_df) > 0:
    scale_factor = 10_000 / len(success_df)
    monthly_baseline = baseline_cost * scale_factor
    monthly_cascade = cascade_cost * scale_factor
    monthly_savings = savings * scale_factor
    
    print(f'\nExtrapolated to 10,000 queries:')
    print(f'  Baseline cost: ${monthly_baseline:.2f}')
    print(f'  Cascade cost: ${monthly_cascade:.2f}')
    print(f'  💵 MONTHLY SAVINGS: ${monthly_savings:.2f}')

print('\n' + '=' * 80)

## 11. Visualizations

In [ ]:
# Model usage distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Pie chart of model usage
model_counts = success_df['model_used'].value_counts()
colors = sns.color_palette('husl', len(model_counts))
ax1.pie(model_counts.values, labels=model_counts.index, autopct='%1.1f%%', 
        colors=colors, startangle=90)
ax1.set_title('Model Selection Distribution', fontsize=14, fontweight='bold')

# Bar chart with costs
model_costs_actual = success_df.groupby('model_used')['total_cost'].sum().sort_values(ascending=False)
ax2.bar(range(len(model_costs_actual)), model_costs_actual.values, color=colors)
ax2.set_xticks(range(len(model_costs_actual)))
ax2.set_xticklabels([m.split('/')[-1] for m in model_costs_actual.index], rotation=45, ha='right')
ax2.set_ylabel('Total Cost ($)', fontsize=12)
ax2.set_title('Total Cost by Model', fontsize=14, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / f'model_distribution_{timestamp}.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'✓ Saved: model_distribution_{timestamp}.png')

In [ ]:
# Cascade behavior analysis
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Cascade vs Direct routing
cascade_counts = success_df['cascaded'].value_counts()
axes[0, 0].pie(cascade_counts.values, 
               labels=['Direct Routing', 'Cascaded'] if not cascade_counts.index[0] else ['Cascaded', 'Direct Routing'],
               autopct='%1.1f%%', colors=['#66c2a5', '#fc8d62'], startangle=90)
axes[0, 0].set_title('Cascade vs Direct Routing', fontsize=14, fontweight='bold')

# 2. Draft acceptance rate
draft_counts = success_df[success_df['cascaded']]['draft_accepted'].value_counts()
if len(draft_counts) > 0:
    axes[0, 1].pie(draft_counts.values,
                   labels=['Rejected', 'Accepted'] if not draft_counts.index[0] else ['Accepted', 'Rejected'],
                   autopct='%1.1f%%', colors=['#e78ac3', '#8da0cb'], startangle=90)
    axes[0, 1].set_title('Draft Acceptance Rate (Cascaded Queries Only)', fontsize=14, fontweight='bold')
else:
    axes[0, 1].text(0.5, 0.5, 'No cascaded queries', ha='center', va='center', fontsize=12)
    axes[0, 1].set_title('Draft Acceptance Rate', fontsize=14, fontweight='bold')

# 3. Latency distribution by model
model_latencies = [success_df[success_df['model_used'] == model]['latency_ms'].values 
                   for model in model_counts.index]
axes[1, 0].boxplot(model_latencies, labels=[m.split('/')[-1] for m in model_counts.index])
axes[1, 0].set_ylabel('Latency (ms)', fontsize=12)
axes[1, 0].set_title('Latency Distribution by Model', fontsize=14, fontweight='bold')
axes[1, 0].tick_params(axis='x', rotation=45)
axes[1, 0].grid(axis='y', alpha=0.3)

# 4. Cost per sample distribution
axes[1, 1].hist(success_df['total_cost'], bins=30, color='#a6d854', edgecolor='black', alpha=0.7)
axes[1, 1].axvline(success_df['total_cost'].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: ${success_df["total_cost"].mean():.6f}')
axes[1, 1].set_xlabel('Cost per Sample ($)', fontsize=12)
axes[1, 1].set_ylabel('Frequency', fontsize=12)
axes[1, 1].set_title('Cost Distribution per Sample', fontsize=14, fontweight='bold')
axes[1, 1].legend()
axes[1, 1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / f'cascade_behavior_{timestamp}.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'✓ Saved: cascade_behavior_{timestamp}.png')

In [ ]:
# Task-wise analysis
task_stats = success_df.groupby('router_task').agg({
    'sample_id': 'count',
    'total_cost': 'mean',
    'latency_ms': 'mean',
    'is_correct': 'mean'
}).rename(columns={
    'sample_id': 'count',
    'total_cost': 'avg_cost',
    'latency_ms': 'avg_latency',
    'is_correct': 'accuracy'
}).sort_values('count', ascending=False).head(15)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# 1. Task volume
axes[0].barh(range(len(task_stats)), task_stats['count'].values, color='#66c2a5')
axes[0].set_yticks(range(len(task_stats)))
axes[0].set_yticklabels(task_stats.index, fontsize=9)
axes[0].set_xlabel('Number of Samples', fontsize=12)
axes[0].set_title('Top Tasks by Volume', fontsize=14, fontweight='bold')
axes[0].grid(axis='x', alpha=0.3)

# 2. Average cost per task
axes[1].barh(range(len(task_stats)), task_stats['avg_cost'].values, color='#fc8d62')
axes[1].set_yticks(range(len(task_stats)))
axes[1].set_yticklabels(task_stats.index, fontsize=9)
axes[1].set_xlabel('Average Cost ($)', fontsize=12)
axes[1].set_title('Average Cost by Task', fontsize=14, fontweight='bold')
axes[1].grid(axis='x', alpha=0.3)

# 3. Accuracy per task
if 'accuracy' in task_stats.columns:
    axes[2].barh(range(len(task_stats)), task_stats['accuracy'].values * 100, color='#8da0cb')
    axes[2].set_yticks(range(len(task_stats)))
    axes[2].set_yticklabels(task_stats.index, fontsize=9)
    axes[2].set_xlabel('Accuracy (%)', fontsize=12)
    axes[2].set_title('Accuracy by Task', fontsize=14, fontweight='bold')
    axes[2].grid(axis='x', alpha=0.3)
    axes[2].set_xlim(0, 100)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / f'task_analysis_{timestamp}.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'✓ Saved: task_analysis_{timestamp}.png')

In [ ]:
# Cost comparison visualization
fig, ax = plt.subplots(figsize=(10, 6))

# Compare baseline vs cascade
comparison_data = {
    'Always Best Model': baseline_cost,
    'Cascade Flow': cascade_cost,
    'Savings': savings
}

colors_bar = ['#e78ac3', '#66c2a5', '#ffd92f']
bars = ax.bar(comparison_data.keys(), comparison_data.values(), color=colors_bar, edgecolor='black', linewidth=1.5)

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'${height:.6f}',
            ha='center', va='bottom', fontsize=12, fontweight='bold')

ax.set_ylabel('Cost ($)', fontsize=14)
ax.set_title(f'Cost Comparison: Cascade vs Always-Best Model\n({savings_pct:.1f}% Savings)', 
             fontsize=16, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / f'cost_savings_{timestamp}.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'✓ Saved: cost_savings_{timestamp}.png')

## 12. Final Summary & Export

In [ ]:
# Create summary report
summary_report = {
    'experiment_info': {
        'timestamp': timestamp,
        'total_samples': len(test_df),
        'successful_samples': len(success_df),
        'failed_samples': len(results_df) - len(success_df),
        'cascade_models': [m.name for m in cascade_models],
    },
    'cascade_behavior': {
        'cascaded_queries': int(cascaded_count),
        'draft_accepted': int(draft_accepted_count),
        'direct_routing': int(len(success_df) - cascaded_count),
        'cascade_rate': float(cascaded_count / len(success_df) * 100),
        'draft_acceptance_rate': float(draft_accepted_count / cascaded_count * 100) if cascaded_count > 0 else 0,
    },
    'performance': {
        'total_cost': float(total_cost),
        'avg_cost_per_sample': float(avg_cost),
        'avg_latency_ms': float(avg_latency),
        'total_tokens': int(total_tokens),
    },
    'cost_savings': {
        'baseline_cost': float(baseline_cost),
        'cascade_cost': float(cascade_cost),
        'absolute_savings': float(savings),
        'percentage_savings': float(savings_pct),
    },
    'model_usage': model_counts.to_dict(),
}

# Add accuracy if available
if 'is_correct' in success_df.columns:
    summary_report['accuracy'] = {
        'contains_ground_truth': float(success_df['is_correct'].sum() / len(success_df) * 100),
        'exact_match': float(success_df['exact_match'].sum() / len(success_df) * 100),
    }

# Save summary
summary_path = OUTPUT_DIR / f'experiment_summary_{timestamp}.json'
with open(summary_path, 'w') as f:
    json.dump(summary_report, f, indent=2)

print('=' * 80)
print('EXPERIMENT COMPLETE')
print('=' * 80)
print(f'\nFiles saved to: {OUTPUT_DIR}')
print(f'  - Results CSV: cascade_results_{timestamp}.csv')
print(f'  - Results JSON: cascade_results_{timestamp}.json')
print(f'  - Summary: experiment_summary_{timestamp}.json')
print(f'  - Visualizations: *.png')
print('\n' + '=' * 80)

# Display summary
print('\nEXPERIMENT SUMMARY:')
print(json.dumps(summary_report, indent=2))